In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import sys
from pathlib import Path

# Project directories
BASE = Path('/content/drive/MyDrive/ML-Project')
RBG_DIR = BASE / 'rbg'

# Locate RBG source directory
RBG_SRC = next(
    (p for p in RBG_DIR.rglob('src') if (p / 'rbg').is_dir()),
    None
)

if RBG_SRC is None:
    raise FileNotFoundError("RBG source directory not found.")

# Add RBG to Python path
if str(RBG_SRC) not in sys.path:
    sys.path.insert(0, str(RBG_SRC))

# Temporary fix for Utilities import issue
utilities_init = RBG_SRC / 'rbg' / 'Utilities' / '__init__.py'

content = utilities_init.read_text(encoding='utf-8')
content = content.replace(
    "from .Matrix_design import matrix_design\n",
    ""
)
utilities_init.write_text(content, encoding='utf-8')

print("RBG is available!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RBG is available!


In [ ]:
# Verificar todos os diretórios candidatos para a biblioteca rbg no seu Drive
from pathlib import Path
import sys

BASE = Path('/content/drive/MyDrive/ML-Project')
RBG_DIR = BASE / 'rbg'

print("=== Buscando diretórios de código 'rbg' no seu Drive ===")
candidatos = []
for p in RBG_DIR.rglob('src'):
    if (p / 'rbg').is_dir():
        candidatos.append(p)

for idx, caminho in enumerate(candidatos):
    # Mostra o caminho completo e se ele está no sys.path atual
    status = "(Ativo no sys.path)" if str(caminho) in sys.path else ""
    print(f"[{idx}] {caminho} {status}")

if not candidatos:
    print("Nenhuma pasta 'src/rbg' foi localizada no diretório rbg.")

=== Buscando diretórios de código 'rbg' no seu Drive ===
[0] /content/drive/MyDrive/ML-Project/rbg/src (Ativo no sys.path)


In [ ]:
#imports:
!pip install -q ucimlrepo "altair>=5" vl-convert-python
import altair as alt
import numpy as np
import requests
import tarfile
import io
import pandas as pd

from rbg.Utilities.crossvalidation import train_test_split
from rbg.ModelSelection.StratifiedKFold import StratifiedKFold
from rbg.ModelSelection.GridSearch import GridSearch
from rbg.Preprocessing.Standard import Standard
from rbg.Classification.KNN import KNN
from rbg.Utilities.ClassificationMetrics import Accuracy, F1Score
from rbg.Unsupervised.PCA import PCA

In [ ]:
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

CLASS_COLORS = {
    "DERMASON": "#0173B2",
    "SIRA":     "#DE8F05",
    "SEKER":    "#029E73",
    "HOROZ":    "#D55E00",
    "CALI":     "#CC78BC",
    "BARBUNYA": "#CA9161",
    "BOMBAY":   "#FBAFE4",
}

def stylize(chart):
    return (
        chart
        .configure_axis(labelFontSize=10, titleFontSize=11, gridColor="#E5E5E5")
        .configure_legend(symbolOpacity=1, labelFontSize=10, titleFontSize=11)
        .configure_title(fontSize=14)
    )

In [ ]:
from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

display(df[:5])
display(df['Class'].value_counts())

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


,count
Class,
DERMASON,3546
SIRA,2636
SEKER,2027
HOROZ,1928
CALI,1630
BARBUNYA,1322
BOMBAY,522


# Processamento dos dados usando PCA e KNN
## Pré-processamento e Prevenção de Vazamento de Dados (Data Leakage)

- **Padronização das Variáveis** ($Z$-score)**:** Tanto o PCA quanto o KNN dependem diretamente da escala geométrica dos atributos numéricos. Sem a padronização, atributos com ampla variação escalar (como ```Area```, que pode chegar a mais de 1000) dominariam desproporcionalmente o cálculo das distâncias e a orientação dos componentes principais em relação a atributos delimitados (como ```Solidity```, que se situa no intervalo $(0, 1]$).

- **Isolamento Estrito dos Dados de Teste:** O dataset é dividido inicialmente em conjuntos de treino e teste pela função ```train_test_split```. O conjunto de teste é completamente reservado antes de qualquer cálculo estatístico.

- **Prevenção de Vazamento de Informação:** A média ($\mu$) e o desvio padrão ($\sigma$) das variáveis são calculados exclusivamente sobre o conjunto de treino via ```scaler.fit(X_train)```. As transformações (```scaler.transform()```) em ```X_train``` e ```X_test``` utilizam unicamente os parâmetros extraídos do treino, impedindo que a distribuição estatística do conjunto de teste contamine o aprendizado do modelo.

In [ ]:
# Separação entre dados de treino e teste:
X = df.drop('Class', axis=1).values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = True, random_state=42)

In [ ]:
# Normalização dos dados:
standard_scaler = Standard()
standard_scaler.fit(X_train)
X_train = standard_scaler.transform(X_train)
# Para evitar vazamento de dados mas manter a padronização entre todos os dados,
# normalizamos os dados do X de teste também pela média e dp dos dados de treino:
X_test = standard_scaler.transform(X_test)

## Redução de Dimensionalidade com PCA (*Principal Component Analysis*)

Existem alguns motivos para aplicarmos o PCA antes do KNN, dentre eles:

- **Maldição da Dimensionalidade (*Curse of Dimensionality*):** Em espaços de alta dimensão, as distâncias entre pares de pontos tendem a se igualar, o que diminui a capacidade da métrica euclidiana usada no KNN. O PCA projeta os dados em um subespaço de menor dimensão, tornando as métricas de distância do KNN mais significativas e computacionalmente eficientes.

- **Eliminação de Colinearidade:** Atributos altamente correlacionados (como ```Area```, ```Perimeter```, ```MajorAxisLength``` e ```MinorAxisLength```) adicionam redundância. O PCA projeta os dados em eixos ortogonais, ou seja, pouco correlacionados, de forma que features "redundantes" acabam não sendo utilizadas.

- **Seleção por Variância Acumulada:** Em vez de escolher um número arbitrário de componentes, define-se uma meta de retenção da variância total de $95\%$. Essa escolha preserva a maior parte da informação estatística essencial do conjunto de dados, descartando apenas variações de baixo impacto e ruídos geométricos de alta dimensão.

- **Integridade do Pipeline de PCA:** A decomposição SVD e a matriz de autovetores são aprendidas apenas sobre o treino padronizado (```pca.fit(X_train_scaled)```). O conjunto de teste padronizado é projetado utilizando a mesma base ortogonal já calculada (```pca.transform(X_test_scaled)```).

In [ ]:
pca = PCA(n_components=0.9) # variância acumulada desejada
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Dimensão original das features: {X_train.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Dimensão original das features: 16
Componentes mantidos pelo PCA: 4
Variância acumulada: 95.04%


## Classificação com KNN (K-Nearest Neighbors)

- **Mecanismo de Decisão:** O KNN atribui a classe de uma nova amostra com base na votação por maioria simples de seus $k$ vizinhos mais próximos no espaço projetado pelo PCA. A ideia de usar GridSearch para decidir qual o melhor hiperparâmetro $k$ é porque sabemos que ele não pode atingir um valor muito grande, pois um valor menor para esse hiperparâmetro faz com que o algoritmo capture melhor padrões locais, o que é essencial para a classificação de um dado novo. Portanto, decidir qual o $k$ que, nesse caso, aumenta nossa métrica é indispensável para fazermos previsões melhores.
- **Pipeline de Seleção de Hiperparâmetros (`GridSearch`):**
  - Validade da Validação Cruzada (`StratifiedKFold`): Aplicamos a validação cruzada sobre os dados de treino com menos features (`X_train_pca`), garantindo que a proporção das classes de feijões seja preservada em cada fold.
  - Uso de $k$ Ímpares: A busca no GridSearch limita a escolha do hiperparâmetro $k$ a valores ímpares, prevenindo cenários de empate na votação da vizinhança.
  - Métrica de Desempenho: Configuramos `greater_is_better=True` no GridSearch para otimizar métricas diretas de classificação, no nosso caso, a F1 Score, que foi utilizada justamente devido ao desbalanceamento de classes.


In [ ]:
_, X_mini, _, y_mini = train_test_split(X_train_pca, y_train, test_size=1000)

valores_de_k = [3, 5, 7, 9]
grid = GridSearch(
    model=KNN,
    param_grid={'k': valores_de_k},
    metrics=F1Score,
    greater_is_better=True,
    cv=StratifiedKFold(k=5)
)

#grid.fit(X_train_pca, y_train)
grid.fit(X_mini, y_mini)
print("Melhor hiperparâmetro encontrado:", grid.best_params_)
print(f"F1 Score médio na validação cruzada: {grid.best_score_ * 100:.2f}%")

Melhor parâmetro encontrado: {'k': 9}
F1 Score médio na validação cruzada: 88.65%


## Avaliação Final do Modelo:
O modelo KNN otimizado e reajustado no treino completo é testado uma única vez sobre `X_test_pca`, gerando métricas de generalização sem viés.

In [ ]:
y_pred_test = grid.predict(X_mini)
test_acc = Accuracy(y_test, y_pred_test)
f1score = F1Score(y_test, y_pred_test)
print(f"F1 Score média nos conjuntos: {np.mean(f1score) * 100:.2f}%")
print(f"Acurácia final no conjunto de teste: {test_acc * 100:.2f}%")

F1 Score média nos conjuntos: 88.82%
Acurácia Final no Conjunto de Teste: 88.58%


## Mais testes com o PCA

Para de fato enxergar como o KNN funciona, podemos usar o PCA para diminuir nossos dados a apenas dois features e plotar os dados em um espaço $\mathbb{R}^2$.

In [ ]:
pca = PCA(n_components=2)
pca.fit(X_train)

X_train_pca2d = pca.transform(X_train)
X_test_pca2d = pca.transform(X_test)

print(f"Dimensão original das features: {X_train.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Dimensão original das features: 16
Componentes mantidos pelo PCA: 2
Variância acumulada: 81.96%


In [ ]:
import altair as alt
import pandas as pd
import numpy as np

# montagem do dataframe com os dois principais componentes determinados pelo pca:
df_pca_plot = pd.DataFrame({"PC1": X_train_pca2d[:, 0],"PC2": X_train_pca2d[:, 1],"Class": y_train})
# escala de cores com dicionário CLASS_COLORS:
color_scale = alt.Scale(domain=list(CLASS_COLORS.keys()),range=list(CLASS_COLORS.values()))
# gráfico de dispersão com altair:
chart = (
    alt.Chart(df_pca_plot)
    .mark_circle(size=30, opacity=0.7)
    .encode(
        x=alt.X("PC1:Q", title="Componente Principal 1 (PC1)"),
        y=alt.Y("PC2:Q", title="Componente Principal 2 (PC2)"),
        color=alt.Color("Class:N", scale=color_scale, title="Classe"),
        tooltip=[
            "Class:N",
            alt.Tooltip("PC1:Q", format=".2f"),
            alt.Tooltip("PC2:Q", format=".2f")
        ]
    )
    .properties(
        title="Dispersão das Amostras com os Dois Principais Componentes",
        width=650,
        height=450
    )
)

# exibição do gráfico final:
chart_stylized = stylize(chart)
display(chart_stylized)

alt.Chart(...)

Analisando esse gráfico, podemos entender porque a acurácia e a F1 Score média não tenham alcançado um valor tão alto: há bastante ruído nos dados. Algumas nuvens de dados se mesclam bastante, como 'CALI' e 'BARBUNYA', além de 'SIRA', que parece estar em uma interseção entre outras cinco classes. Para essa quantidade de ruído, a F1 Score e Acurácia de cerca de 90% nos dados de teste é um valor bastante satisfatório.

# Árvores

In [ ]:
from rbg.DecisionTree import ClassificationTree, Bagging, RandomForest

import numpy as np

import matplotlib.pyplot as plt

import pandas as pd

from rbg.Utilities import Gini, train_test_split, error_rate

from rbg.ModelSelection import GridSearch, KFold




Faremos agora o uso de árvores para prever os nossos dados. Primeiro, faremos uma separação entre dados de treino e de teste. Em seguida, aplicaremos às árvores de regressão com e sem poda, bagging e Random Forest os nossos dados, afim de que tenhamos uma previsão. Começaremos primeiro com todos os dados usando os valores padrão, a fim de ilustrar o uso da biblioteca, e também  faremos apenas um plot gráfico comparativo entre a árvore com e sem poda e tiraremos o erro entre essas árvores. Além do erro calculado com o MSE, também faremos o OOB da random e bagging tree.


* Validação:

Para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, o melhor hiperparâmetro para eles.

* Comparação de Modelos:

Com os modelos ajustados pelos seus "hiperparâmetros mais ajustados", iremos fazer uma validação cruzada cmo os dados de treino para decidir qual o melhor modelo.

* Avaliação Final:

Dado o modelo escolhido, iremos fazer a predição dos dados de teste, e tomar seu erro.

* Interpretação dos resultados:

Parte destinada a mais considerações sobre escolhas de métricas e trade-off do viés e da variância.



In [ ]:
def plot_arvore(tree, feature_names):


    info = tree.get_tree_structure_info()


    left = info['children_left']
    right = info['children_right']
    feature = info['feature']
    threshold = info['threshold']
    value = info['value']

    # 1. Dicionário para armazenar as posições (x, y) de cada nó
    pos = {}

    def posicao(node, x=0, depth=0):
        if node == -1:
            return x

        if left[node] != -1:
            x = posicao(left[node], x, depth + 1)

        pos[node] = (x, -depth)
        x += 1.0

        # Visita sub-árvore direita
        if right[node] != -1:
            x = posicao(right[node], x, depth + 1)

        return x
    posicao(0, x=0, depth=0)

    # 2. Configura a figura do Matplotlib
    plt.figure(figsize=(14, 8))
    ax = plt.gca()
    ax.axis('off')

    for node in pos:
        if left[node] != -1 and left[node] in pos:
            x_parent, y_parent = pos[node]
            x_left, y_left = pos[left[node]]
            plt.plot([x_parent, x_left], [y_parent, y_left], color='gray', lw=1.5, zorder=1)

        if right[node] != -1 and right[node] in pos:
            x_parent, y_parent = pos[node]
            x_right, y_right = pos[right[node]]
            plt.plot([x_parent, x_right], [y_parent, y_right], color='gray', lw=1.5, zorder=1)

    # 4. Desenha os nós (caixas de texto)
    for node, (x, y) in pos.items():
        is_leaf = (left[node] == -1 and right[node] == -1)

        if is_leaf:
            # Formatação para nós folha (previsão)
            val = value[node]
            if isinstance(val, (np.ndarray, list)):
                val_str = f"Pred: {np.round(val, 2)}"
            else:
                val_str = f"Pred: {val:.2f}" if isinstance(val, (int, float, np.number)) else f"Pred: {val}"

            # Caixa vermelha clara para folhas
            bbox_style = dict(boxstyle="round,pad=0.6", fc="#ffcccc", ec="black", lw=1.2)
        else:
            # Formatação para nós de decisão (feature + limiar)
            feat_idx = feature[node]
            if feature_names is not None and feat_idx < len(feature_names):
                feat_name = feature_names[feat_idx]
            else:
                feat_name = f"X[{feat_idx}]"

            thresh = threshold[node]
            val_str = f"{feat_name}\n≤ {thresh:.2f}"

            # Caixa azul clara para nós de decisão
            bbox_style = dict(boxstyle="round,pad=0.6", fc="#cce5ff", ec="black", lw=1.2)

        plt.text(x, y, val_str, ha='center', va='center', fontsize=9, bbox=bbox_style, zorder=2)

    plt.title("Visualização da Estrutura da Árvore de Decisão", fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:


X = df[FEATURES].to_numpy()

y = df[TARGET].to_numpy()



In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X,y,random_state = 42)

In [ ]:
arvore = ClassificationTree()

arvore.fit(X_treino,y_treino)

y_previsto = arvore.predict(X_teste)

erro_arvore_sem_poda = error_rate(y_previsto,y_teste)

print("erro da arvore sem poda:",erro_arvore_sem_poda)

erro da arvore sem poda: 0.09291222915901576


In [ ]:
arvore_poda = ClassificationTree()

arvore_poda.fit(X_treino,y_treino, prune = True)

y_previsto_poda = arvore_poda.predict(X_teste)

erro_arvore_com_poda = error_rate(y_previsto_poda,y_teste)

print("erro da arvore com poda:",erro_arvore_com_poda)

info = arvore_poda.get_tree_structure_info()

In [ ]:
arvorebag = Bagging()

arvorebag.fit(X_treino,y_treino)

y_previstobag = arvorebag.predict(X_teste)

erro_arvore_bag = error_rate(y_previstobag,y_teste)

print("erro da bagging :",erro_arvore_bag)

In [ ]:
arvore_random = RandomForest()

arvore_random.fit(X_treino,y_treino)

y_previsto_random = arvore_random.predict(X_teste)

erro_arvore_random = error_rate(y_previsto_random,y_teste)

print("erro da arvore random:",erro_arvore_random)

Plot da árvore com poda:

In [ ]:

plot_arvore(arvore_poda, feature_names=FEATURES)

Plot da árvore sem poda:

In [ ]:

plot_arvore(arvore, feature_names=FEATURES)

## Validação:

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) tem um custo grande para serem montadas de acordo com o algoritmo implementado (a escolha de melhor splitter que passa por todos os valores, etc), e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada, usaremos apenas os preditores:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split (e o nosso "criterion_evaluation" será por tanto error_rate)

Dentro de criterion_choose, também iremos apenas avaliar a moda já que média para problema de classificação não faz muito sentido, e mediana, apesar dos feitojões terem sido numerados, não tem de fato " uma ordem" para tirarmos a mediana.Também deixaremos de fora "min" e "max" pois não necessariamente " as variáveis dummies extremas" são um bom recorte da classifiação.

Também, nos limitaremos a apenas Gini e entropia para serem os critérios de cálculo de impureza.





In [ ]:
#Parametros:

p_grid = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2,10,50],
    'min_impurity_decrease':[0.01, 0.1, 1,10],
    'criterion_split':[ 'Standard','entropy','classif_error'],
    'criterion_choose':['Standard']

}

grid_arvore = GridSearch( model = ClassificationTree(), param_grid = p_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )

In [ ]:
grid_arvore.fit(X_treino,y_treino)

Agora iremos plotar os parâmetros escolhidos pela nossa árvore.

In [ ]:
print(grid_arvore.best_params_)

Para árvores do tipo baggin, que por terem sido implementadas apenas com numpy (logo, sem a biblioteca de paralelização), fazem as árvores localmente. Logo, fazer apenas uma árvore de bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda. Mudaremos apenas pro bagging a quantidade de árvores utilizadas. Também faremos uma busca de hiperparâmetros dos elementos visto anteriormente, mas com ainda menos elementos, para ser viável computacionalmente.

In [ ]:
pbaggin_grid = {
    'max_depth': [10],
    'min_samples_split': [10],
    'min_impurity_decrease':[1,10],
    'criterion_split':[ 'Standard','entropy'],
    'criterion_choose':['Standard'],
    'n_trees':[10, 20]

}

grid_bagging = GridSearch( model = Bagging(), param_grid = pbaggin_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )

In [ ]:
grid_bagging.fit(X_treino,y_treino)

In [ ]:
print(grid_bagging.best_params_)

Para árvores do tipo baggin, que por terem sido implementadas apenas com numpy (logo, sem a biblioteca de paralelização), fazem as árvores localmente. Logo, fazer apenas uma árvore de bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda. Utilizaremos apenas o teste de 100 árvores, que é o padrão do baggin, e para ser viável computacionalmente.

In [ ]:
p_baggin_grid = p_grid = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2,10,50],
    'min_impurity_decrease':[0.01, 0.1, 1,10],
    'criterion_split':[ 'Standard','entropy'],
    'criterion_choose':['Standard'],
    'n_trees':[100]

}

grid_bagging = GridSearch( model = Bagging(), param_grid = p_baggin_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )

In [ ]:
grid_bagging.fit(X_treino,y_treino)

In [ ]:
print(grid_bagging.best_params_)

Similarmente, também vamos testar, para random forest, apenas 100 árvores e com o critério de escolha de quantidade de parâmetros ser piso da raiz quadrada do número total de preditores. Como a nosso dataset tem apenas 7 preditores, temos que sempre dois preditores serão escolhidos.

In [ ]:


p_random_grid = p_grid = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2,10,50],
    'min_impurity_decrease':[0.01, 0.1, 1,10],
    'criterion_split':[ 'Standard','entropy'],
    'criterion_choose':['Standard'],
    'n_trees':[100],
    'factor_selection': [np.sqrt]

}

grid_Random = GridSearch( model = RandomForest(), param_grid = p_random_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )



In [ ]:
grid_Random.fit(X_treino,y_treino)

# Seleção de Modelo


Para a seleção de modelos, iremos fazer um K-Fold usando "os campeões dos hiperparâmetros", treinaremos o dataset neles, e escolheremos aquele que teve menor erro de acordo com o MSE

In [ ]:

param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

print(param_r)

In [ ]:
param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

bagging = Bagging(**param_bagging)
arvore_pos_grid = ClassificationTree(**param_arvore)
random = RandomForest(**param_r)

In [ ]:



k_fold = KFold(random_state = 41, k = 5)
k = k_fold.split(X_treino.copy())
lista_erros_arvore =[]
lista_erros_bag=[]
lista_erros_random =[]


for _, (treino_k, teste_k) in enumerate(k):
    X_treino_k, X_teste_k = X_treino[treino_k], X_treino[teste_k]
    y_treino_k, y_teste_k = y_treino[treino_k], y_treino[teste_k]

    random.fit(X_treino_k,y_treino_k)
    bagging.fit(X_treino_k,y_treino_k)
    arvore_pos_grid.fit(X_treino_k,y_treino_k)
    y_pred_random = random.predict(X_teste_k)
    y_pred_bagging = bagging.predict(X_teste_k)
    y_pred_arvore = arvore_pos_grid.predict(X_teste_k)
    lista_erros_arvore.append(error_rate(y_teste_k, y_pred_arvore))
    lista_erros_bag.append(error_rate(y_teste_k, y_pred_bagging))
    lista_erros_random.append(error_rate(y_teste_k, y_pred_random))

erro_medio_a = np.mean(lista_erros_arvore)
erro_medio_b = np.mean(lista_erros_bag)
erro_medio_r = np.mean(lista_erros_random)


print("erro medio da arvore:",erro_medio_a)
print("erro medio do bagging:",erro_medio_b)
print("erro medio do random forest:",erro_medio_r)

# Avaliação Final

Com o nosso modelo ganhador, iremos treinar todos os dados de treino, para aplicar aos nossos dados de teste,
e depois tirar o erro com o MSE

In [ ]:
if erro_medio_b >= erro_medio_a and erro_medio_b >= erro_medio_r:
    bagging.fit(X_treino,y_treino)
    predicao_final = bagging.predict(X_teste)
elif erro_medio_a >= erro_medio_b and erro_medio_b >= erro_medio_r:
    arvore_pos_grid.fit(X_treino,y_treino)
    predicao_final = arvore_pos_grid.predict(X_teste)
else:
    arvore_pos_grid.fit(X_treino,y_treino)
    predicao_final = arvore_pos_grid.predict(X_teste)

print("previsao final é:",predicao_final)

print("Erro do modelo:",error_rate(predicao_final,y_teste))


Como o modelo final também foi um que tem o método OOB, podemos plotar esse erro OOB:

# Interpretação dos Resultados:

Uma vez que o modelo ....

Falar sobre:

viés
variância
métrica escolhida para fazer o SearchGrid


Dado que tivemos uma transformação do X_treino com o PCA e X_teste também, iremos